<a href="https://colab.research.google.com/github/kimgayeon430/IOT/blob/main/Week_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 12: Language Models (GloVe → Transformer/BERT)로 추천 만들기

### 공통: 환경 설정 & Amazon 리뷰 데이터 로딩

## 1. 환경 설정과 데이터 준비 (공통)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional
import shutil
import urllib.request
from math import log2

SEED = 42
rng = np.random.default_rng(SEED)

DATA_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Magazine_Subscriptions.jsonl.gz"
DATA_DIR = Path("./data")
LOCAL_GZ_PATH = DATA_DIR / "Magazine_Subscriptions.jsonl.gz"

N_SAMPLES = 10000
MIN_INTERACTIONS_USER = 2
MIN_INTERACTIONS_ITEM = 1


def download_if_needed(url: str, local_path: Path, timeout: int = 10) -> None:
    """데이터 파일이 없을 때만 내려받습니다."""
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists() and local_path.stat().st_size > 0:
        print(f"Using cached file: {local_path}")
        return

    tmp_path = local_path.with_suffix(local_path.suffix + ".tmp")
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response, open(tmp_path, "wb") as f:
            shutil.copyfileobj(response, f)
        tmp_path.replace(local_path)
        print(f"Dataset saved to: {local_path}")
    except Exception:
        if tmp_path.exists():
            tmp_path.unlink()
        raise


def load_local_jsonl_gz(local_path: Path, n_samples: Optional[int] = None) -> pd.DataFrame:
    """gzip JSONL 형식의 Amazon 리뷰 데이터를 DataFrame으로 읽습니다."""
    if n_samples is not None:
        reader = pd.read_json(local_path, lines=True, compression="gzip", chunksize=n_samples)
        df = next(reader).copy()
    else:
        df = pd.read_json(local_path, lines=True, compression="gzip")
    print(f"Loaded rows: {len(df):,}")
    return df


def _make_toy_amazon_like(
    n_users: int = 200,
    n_items: int = 300,
    n_rows: int = 5000,
    seed: int = SEED,
) -> pd.DataFrame:
    """Amazon 리뷰 데이터와 같은 컬럼 구조를 가진 예제 데이터를 만듭니다."""
    rng_local = np.random.default_rng(seed)
    users = [f"U{u:04d}" for u in range(n_users)]
    items = [f"I{i:04d}" for i in range(n_items)]

    title_pool = [
        "Great value",
        "Would buy again",
        "Just okay",
        "Not recommended",
        "Fast shipping",
    ]
    text_pool = [
        "Great quality and fast shipping.",
        "Not what I expected. The material feels cheap.",
        "Works as described. Good value for the price.",
        "Terrible experience. Would not recommend.",
        "Decent product, but packaging was damaged.",
    ]

    rows = []
    for _ in range(n_rows):
        user_id = rng_local.choice(users)
        item_id = rng_local.choice(items)
        rating = int(rng_local.integers(1, 6))
        title = rng_local.choice(title_pool)
        review_text = rng_local.choice(text_pool)
        helpful = int(rng_local.integers(0, 10))
        total = helpful + int(rng_local.integers(0, 5))
        ts = pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(rng_local.integers(0, 365)))
        rows.append((user_id, item_id, rating, title, review_text, helpful, total, ts))

    return pd.DataFrame(
        rows,
        columns=[
            "user_id",
            "item_id",
            "rating",
            "review_title",
            "review_text",
            "helpful_votes",
            "total_votes",
            "timestamp",
        ],
    )


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Amazon 리뷰 데이터의 여러 컬럼명을 실습용 공통 컬럼명으로 맞춥니다."""
    def pick(*candidates):
        for col in candidates:
            if col in df.columns:
                return col
        return None

    user_col = pick("user_id", "customer_id", "reviewerID", "reviewer_id", "reviewerId")
    item_col = pick("parent_asin", "asin", "product_id", "item_id")
    rating_col = pick("rating", "star_rating", "overall", "stars")
    text_col = pick("text", "review_body", "reviewText", "review_text", "review")
    title_col = pick("title", "review_headline", "summary", "review_title")
    helpful_col = pick("helpful_vote", "helpful_votes", "helpful")
    total_col = pick("total_votes", "total")
    time_col = pick("timestamp", "review_date", "unixReviewTime", "date")

    if user_col is None or item_col is None or rating_col is None or text_col is None:
        raise ValueError(
            "필수 컬럼(user/item/rating/text)을 찾지 못했습니다. "
            f"user={user_col}, item={item_col}, rating={rating_col}, text={text_col}"
        )

    out = pd.DataFrame({
        "user_id": df[user_col].astype(str),
        "item_id": df[item_col].astype(str),
        "rating": pd.to_numeric(df[rating_col], errors="coerce"),
        "review_text": df[text_col].fillna("").astype(str),
    })
    out["review_title"] = df[title_col].fillna("").astype(str) if title_col is not None else ""

    out["helpful_votes"] = 0
    out["total_votes"] = 0
    if helpful_col is not None:
        first_helpful = df[helpful_col].dropna().iloc[0] if df[helpful_col].notna().any() else None
        if isinstance(first_helpful, (list, tuple)) and len(first_helpful) >= 2:
            out["helpful_votes"] = [
                x[0] if isinstance(x, (list, tuple)) and len(x) >= 1 else 0
                for x in df[helpful_col]
            ]
            out["total_votes"] = [
                x[1] if isinstance(x, (list, tuple)) and len(x) >= 2 else 0
                for x in df[helpful_col]
            ]
        else:
            out["helpful_votes"] = pd.to_numeric(df[helpful_col], errors="coerce").fillna(0).astype(int)
            if total_col is not None:
                out["total_votes"] = pd.to_numeric(df[total_col], errors="coerce").fillna(0).astype(int)
            else:
                out["total_votes"] = out["helpful_votes"]

    if time_col is not None:
        if time_col == "unixReviewTime":
            out["timestamp"] = pd.to_datetime(pd.to_numeric(df[time_col], errors="coerce"), unit="s", errors="coerce")
        else:
            numeric_time = pd.to_numeric(df[time_col], errors="coerce")
            if numeric_time.notna().any():
                median_abs = float(numeric_time.dropna().abs().median())
                if median_abs > 1e14:
                    out["timestamp"] = pd.to_datetime(numeric_time, unit="ns", errors="coerce")
                elif median_abs > 1e11:
                    out["timestamp"] = pd.to_datetime(numeric_time, unit="ms", errors="coerce")
                elif median_abs > 1e9:
                    out["timestamp"] = pd.to_datetime(numeric_time, unit="s", errors="coerce")
                else:
                    out["timestamp"] = pd.to_datetime(df[time_col], errors="coerce")
            else:
                out["timestamp"] = pd.to_datetime(df[time_col], errors="coerce")
    else:
        out["timestamp"] = pd.NaT

    out = out.dropna(subset=["rating"])
    out["rating"] = out["rating"].clip(1, 5).astype(float)
    return out


def k_core_filter(df: pd.DataFrame, min_user: int, min_item: int, max_iters: int = 10) -> pd.DataFrame:
    """사용자와 아이템의 최소 상호작용 수 조건을 반복 적용합니다."""
    filtered = df.copy()
    for _ in range(max_iters):
        before = len(filtered)
        user_counts = filtered["user_id"].value_counts()
        item_counts = filtered["item_id"].value_counts()
        filtered = filtered[filtered["user_id"].isin(user_counts[user_counts >= min_user].index)]
        filtered = filtered[filtered["item_id"].isin(item_counts[item_counts >= min_item].index)]
        if len(filtered) == before:
            break
    return filtered.copy()


def index_ids(df: pd.DataFrame):
    users = df["user_id"].unique().tolist()
    items = df["item_id"].unique().tolist()
    user2idx = {user_id: idx for idx, user_id in enumerate(users)}
    item2idx = {item_id: idx for idx, item_id in enumerate(items)}
    indexed = df.copy()
    indexed["u"] = indexed["user_id"].map(user2idx).astype(int)
    indexed["i"] = indexed["item_id"].map(item2idx).astype(int)
    return indexed, user2idx, item2idx


def leave_one_out_split(df: pd.DataFrame, seed: int = SEED):
    """각 사용자별로 하나의 상호작용을 test set으로 분리합니다."""
    if df.empty:
        raise ValueError("비어 있는 DataFrame은 분리할 수 없습니다.")

    split_df = df.copy()
    if split_df["timestamp"].notna().any():
        split_df = split_df.sort_values(["user_id", "timestamp"])
        test_idx = split_df.groupby("user_id").tail(1).index
    else:
        test_idx = split_df.groupby("user_id", group_keys=False).sample(n=1, random_state=seed).index

    test = split_df.loc[test_idx].copy()
    train = split_df.drop(index=test_idx).copy()
    return train, test


def prepare_interaction_df(raw_or_standard_df: pd.DataFrame, already_standardized: bool = False) -> pd.DataFrame:
    df = raw_or_standard_df.copy() if already_standardized else standardize_columns(raw_or_standard_df)

    for col, default in {"review_title": "", "helpful_votes": 0, "total_votes": 0}.items():
        if col not in df.columns:
            df[col] = default
    if "timestamp" not in df.columns:
        df["timestamp"] = pd.NaT

    df["review_title"] = df["review_title"].fillna("").astype(str)
    df["review_text"] = df["review_text"].fillna("").astype(str)
    df["helpful_votes"] = pd.to_numeric(df["helpful_votes"], errors="coerce").fillna(0).astype(int)
    df["total_votes"] = pd.to_numeric(df["total_votes"], errors="coerce").fillna(0).astype(int)
    df["total_votes"] = np.maximum(df["total_votes"], df["helpful_votes"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    if df["timestamp"].notna().any():
        df = df.sort_values("timestamp").drop_duplicates(subset=["user_id", "item_id"], keep="last")
    else:
        df = df.drop_duplicates(subset=["user_id", "item_id"], keep="last")

    return k_core_filter(df, MIN_INTERACTIONS_USER, MIN_INTERACTIONS_ITEM)


try:
    download_if_needed(DATA_URL, LOCAL_GZ_PATH)
    raw_df = load_local_jsonl_gz(LOCAL_GZ_PATH, N_SAMPLES)
    df = prepare_interaction_df(raw_df, already_standardized=False)
except Exception as exc:
    print("실제 Amazon 리뷰를 사용할 수 없어 예제 데이터로 진행합니다.")
    print("원인:", repr(exc))
    df = prepare_interaction_df(_make_toy_amazon_like(), already_standardized=True)

if df.empty or df["user_id"].nunique() == 0 or df["item_id"].nunique() == 0:
    print("상호작용 데이터가 비어 있어 예제 데이터로 진행합니다.")
    df = prepare_interaction_df(_make_toy_amazon_like(), already_standardized=True)

print("\n[데이터 확인]")
print(df.head())
print(df.shape, df.columns.tolist())
print("#Users:", df["user_id"].nunique(), "#Items:", df["item_id"].nunique())

df, user2idx, item2idx = index_ids(df)
n_users = len(user2idx)
n_items = len(item2idx)

train_df, test_df = leave_one_out_split(df)
if train_df.empty or test_df.empty:
    raise ValueError("train/test 데이터가 비어 있습니다. 데이터 설정을 확인하세요.")

print("\n[Train/Test]")
print(train_df.shape, test_df.shape)
print(f"n_users={n_users}, n_items={n_items}")


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"shape mismatch: y_true{y_true.shape} vs y_pred{y_pred.shape}")
    if y_true.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


train_user_items = train_df.groupby("u")["i"].apply(lambda s: set(map(int, s))).to_dict()
all_items = np.arange(n_items, dtype=int)


def sample_negatives_for_user(u: int, pos_i: int, n_neg: int, rng: np.random.Generator):
    seen = set(train_user_items.get(int(u), set()))
    seen.add(int(pos_i))
    if len(seen) >= n_items:
        return np.array([], dtype=int)

    mask = np.ones(n_items, dtype=bool)
    mask[list(seen)] = False
    pool = all_items[mask]
    size = min(int(n_neg), len(pool))
    return rng.choice(pool, size=size, replace=False)


def dcg(rels):
    return sum(rel / log2(idx + 2) for idx, rel in enumerate(rels))


def ndcg_at_k(ranked_items, ground_truth_item, k: int):
    rels = [1 if int(item) == int(ground_truth_item) else 0 for item in ranked_items[:k]]
    ideal = [1] + [0] * (k - 1)
    denom = dcg(ideal)
    return 0.0 if denom == 0 else dcg(rels) / denom


def evaluate_topk(model, test_df: pd.DataFrame, k: int = 10, n_neg: int = 100, seed: int = SEED):
    rng_eval = np.random.default_rng(seed)
    hits, ndcgs = [], []

    for row in test_df.itertuples(index=False):
        u = int(row.u)
        pos_i = int(row.i)
        negs = sample_negatives_for_user(u, pos_i, n_neg, rng_eval)
        if len(negs) == 0:
            continue

        candidates = np.concatenate([negs, [pos_i]]).astype(int)
        scores = np.asarray(model.predict(u, candidates), dtype=float)
        if scores.shape[0] != candidates.shape[0]:
            raise ValueError("model.predict는 candidates와 같은 길이의 score를 반환해야 합니다.")

        scores = np.nan_to_num(scores, nan=-1e9, posinf=1e9, neginf=-1e9)
        ranked = candidates[np.argsort(-scores)]
        hits.append(1 if pos_i in ranked[:k] else 0)
        ndcgs.append(ndcg_at_k(ranked, pos_i, k))

    if len(hits) == 0:
        return {f"Hit@{k}": 0.0, f"NDCG@{k}": 0.0, "n_eval": 0}

    return {f"Hit@{k}": float(np.mean(hits)), f"NDCG@{k}": float(np.mean(ndcgs)), "n_eval": int(len(hits))}


print("Evaluation utilities ready.")
train_df.head()

Using cached file: data\Magazine_Subscriptions.jsonl.gz
Loaded rows: 10,000

[데이터 확인]
                           user_id     item_id  rating  \
9477  AHR7ZC2G4VF4MEJMXTZANR23OJ6Q  B00005N7PT     4.0   
9476  AHR7ZC2G4VF4MEJMXTZANR23OJ6Q  B00005N7VP     5.0   
620   AGXFIFO4VWNHPAX3VJPRQ2QJSCSQ  B00005N7P8     4.0   
619   AGXFIFO4VWNHPAX3VJPRQ2QJSCSQ  B00005N7TX     5.0   
618   AGXFIFO4VWNHPAX3VJPRQ2QJSCSQ  B00005NIOR     4.0   

                                            review_text  \
9477  Discover is a fun magazine, and a much easier ...   
9476  I can't tell you how many great science fictio...   
620   Granted, it is called 'Catholic Digest' but it...   
619   &quot;Running Times&quot; is for runners who c...   
618   Hardcore runners will read "Runner's World" be...   

                                           review_title  helpful_votes  \
9477                        The lighter side of Science             81   
9476     The proving ground for science fiction writers       

,user_id,item_id,rating,review_text,review_title,helpful_votes,total_votes,timestamp,u,i
5444,AE22KFYQBL52KJR5BWDLFR4IZF4A,B00005NIP7,1.0,"As of today, I am still waiting to receive Tra...",Never got the magazine,5,5,2010-01-30 19:52:11,100,163
5443,AE22KFYQBL52KJR5BWDLFR4IZF4A,B00005N7TG,5.0,I had purchase 3 different magazines on Amazon...,Subscription received,0,0,2010-01-30 19:56:07,100,92
982,AE23XZF2JZLER42SHFHB6BSE5IIA,B00006KFK1,5.0,My wife loves it.,Loves it.,0,0,2014-11-17 15:27:35,454,223
981,AE23XZF2JZLER42SHFHB6BSE5IIA,B0045FEHCS,5.0,Wife loves it!,Wife loves it!,2,2,2017-02-24 11:04:33,454,420
7260,AE25ONEGFWY6QOK2VAW6ZQK3GLQQ,B001U5SPME,5.0,I subscribe to only 1 magazine: Wired. I am an...,"Educational, entertaining far beyond tech stuff",2,2,2013-01-18 22:10:36,237,248


## 2. 실습 목표

1. 사전학습된 GloVe 단어 임베딩을 불러와 리뷰 토큰을 벡터로 변환합니다.
2. 아이템 리뷰를 임베딩으로 요약해 item embedding을 만듭니다.
3. 사용자가 선호한 아이템의 평균 임베딩으로 user embedding을 만들고 추천 점수를 계산합니다.
4. Transformer/BERT 문장 임베딩 방식과 GloVe 기반 방식을 비교합니다.

## 3. 텍스트 전처리와 아이템 문서 만들기

GloVe와 문장 임베딩 실습을 위해 리뷰 제목과 본문을 하나의 텍스트로 합칩니다. GloVe는 단어 단위 사전학습 임베딩이므로 영문자와 숫자 중심의 간단한 토큰화를 적용합니다.

In [ ]:
import re
from numpy.linalg import norm

TEXT_SAMPLE = min(20000, len(train_df))
text_df = train_df.sample(n=TEXT_SAMPLE, random_state=SEED).copy() if TEXT_SAMPLE > 0 else train_df.copy()
texts = (text_df["review_title"].astype(str) + " " + text_df["review_text"].astype(str)).tolist()

token_pat = re.compile(r"[a-zA-Z0-9]+")


def tokenize(text: str):
    tokens = token_pat.findall(str(text).lower())
    return [token for token in tokens if len(token) >= 2]


sentences = [tokenize(text) for text in texts]
sentences = [sentence for sentence in sentences if len(sentence) >= 3]
if len(sentences) == 0:
    sentences = [["empty", "review", "text"]]


def build_item_documents(frame: pd.DataFrame):
    item_doc_df = frame.copy()
    item_doc_df["_doc"] = item_doc_df["review_title"].astype(str) + " " + item_doc_df["review_text"].astype(str)
    item_docs_series = item_doc_df.groupby("i")["_doc"].apply(lambda parts: " ".join(parts)).sort_index()
    items = item_docs_series.index.astype(int).to_numpy()
    docs = item_docs_series.tolist()
    return items, docs


items_for_text, item_docs = build_item_documents(train_df)

print("#sentences:", len(sentences))
print("#item documents:", len(item_docs))
print("example tokens:", sentences[0][:20])

#sentences: 2513
#item documents: 845
example tokens: ['great', 'gift', 'purchased', 'this', 'subscription', 'on', 'prime', 'day', 'it', 'was', 'great', 'value', 'and', 'great', 'gift', 'for', 'my', 'nieces', 'know', 'as']


## 4. 후보 아이템 기반 Top-K 평가 함수

텍스트 임베딩 모델은 일부 아이템 문서만 사용해 학습할 수 있습니다. 아래 함수는 평가 후보를 해당 아이템 집합으로 제한해 Hit@K와 NDCG@K를 계산합니다.

In [ ]:
def evaluate_topk_with_candidates(
    model,
    test_df: pd.DataFrame,
    candidate_items,
    k: int = 10,
    n_neg: int = 100,
    seed: int = SEED,
):
    rng_eval = np.random.default_rng(seed)
    candidate_items = np.unique(np.asarray(candidate_items, dtype=int))
    candidate_set = set(map(int, candidate_items.tolist()))

    if len(candidate_items) == 0:
        return {f"Hit@{k}": 0.0, f"NDCG@{k}": 0.0, "n_eval": 0}

    hits, ndcgs = [], []
    for row in test_df.itertuples(index=False):
        u = int(row.u)
        pos_i = int(row.i)
        if pos_i not in candidate_set:
            continue

        seen = set(train_user_items.get(u, set()))
        neg_pool = np.array([item for item in candidate_items if item not in seen and item != pos_i], dtype=int)
        if len(neg_pool) == 0:
            continue

        size = min(int(n_neg), len(neg_pool))
        negs = rng_eval.choice(neg_pool, size=size, replace=False)
        candidates = np.concatenate([negs, [pos_i]]).astype(int)
        scores = np.asarray(model.predict(u, candidates), dtype=float)
        if scores.shape[0] != candidates.shape[0]:
            raise ValueError("model.predict는 candidates와 같은 길이의 score를 반환해야 합니다.")

        scores = np.nan_to_num(scores, nan=-1e9, posinf=1e9, neginf=-1e9)
        ranked = candidates[np.argsort(-scores)]
        hits.append(1 if pos_i in ranked[:k] else 0)
        ndcgs.append(ndcg_at_k(ranked, pos_i, k))

    if len(hits) == 0:
        return {f"Hit@{k}": 0.0, f"NDCG@{k}": 0.0, "n_eval": 0}

    return {f"Hit@{k}": float(np.mean(hits)), f"NDCG@{k}": float(np.mean(ndcgs)), "n_eval": int(len(hits))}


print("Candidate-based evaluation ready.")

Candidate-based evaluation ready.


## 5. GloVe 사전학습 단어 임베딩 로딩

`gensim.downloader`로 사전학습된 GloVe 모델을 불러와 inference 전용으로 사용합니다.

기본 모델은 `glove-wiki-gigaword-50`입니다.

In [ ]:
# Inference-only GloVe embedding
!pip install gensim

import gensim.downloader as gensim_api


GLOVE_MODEL_NAME = "glove-wiki-gigaword-50"
# 다른 선택지 예시:
# - "glove-wiki-gigaword-100"
# - "glove-wiki-gigaword-200"
# - "glove-wiki-gigaword-300"
# - "glove-twitter-25"
# - "glove-twitter-50"
# - "glove-twitter-100"
# - "glove-twitter-200"

# 최초 실행 시 인터넷으로 모델을 다운로드하고, 이후에는 gensim cache에서 불러옵니다.
# 이 줄에서 반환되는 객체는 gensim.models.KeyedVectors입니다.
w2v = gensim_api.load(GLOVE_MODEL_NAME)
embedding_source = f"pretrained GloVe via gensim-data: {GLOVE_MODEL_NAME}"

print("embedding source:", embedding_source)
print("vocab size:", len(w2v.index_to_key))
print("vector size:", w2v.vector_size)

shown = 0
for word in ["good", "bad", "price", "quality", "love"]:
    if word in w2v:
        print(f"\nMost similar to '{word}':")
        print(w2v.most_similar(word, topn=8))
        shown += 1

if shown == 0 and len(w2v.index_to_key) > 0:
    word = w2v.index_to_key[0]
    print(f"\nMost similar to '{word}':")
    print(w2v.most_similar(word, topn=8))

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


embedding source: pretrained GloVe via gensim-data: glove-wiki-gigaword-50
vocab size: 400000
vector size: 50

Most similar to 'good':
[('better', 0.9284391403198242), ('really', 0.9220623970031738), ('always', 0.9165270924568176), ('sure', 0.903351366519928), ('something', 0.9014206528663635), ('think', 0.8982065320014954), ('way', 0.8953989744186401), ('thing', 0.894504964351654)]

Most similar to 'bad':
[('worse', 0.8878378868103027), ('unfortunately', 0.8650501370429993), ('too', 0.8608258366584778), ('really', 0.8486315011978149), ('little', 0.8427671194076538), ('bit', 0.8359227776527405), ('things', 0.8306117653846741), ('nothing', 0.8246172666549683)]

Most similar to 'price':
[('prices', 0.9086880087852478), ('sales', 0.8435637354850769), ('market', 0.8383352756500244), ('drop', 0.8382893204689026), ('stock', 0.8372058272361755), ('higher', 0.8235951066017151), ('rise', 0.8151090145111084), ('earnings', 0.8133413791656494)]

Most similar to 'quality':
[('excellent', 0.85576331

## 6. GloVe 기반 item/user embedding 추천

아이템 임베딩은 해당 아이템 리뷰에 등장한 GloVe 단어 벡터의 평균으로 만듭니다. 유저 임베딩은 사용자가 높게 평가한 아이템 임베딩의 평균으로 만들고, 유저와 아이템의 코사인 유사도를 추천 점수로 사용합니다.

In [ ]:
MAX_ITEMS_W2V = min(3000, len(items_for_text))
items_for_w2v = items_for_text[:MAX_ITEMS_W2V]
docs_for_w2v = item_docs[:MAX_ITEMS_W2V]


def embed_tokens_avg(tokens):
    # 문장 (tokens) 내 특정 단어 (token)이 단어 사전 (w2v)에 있으면 해당 벡터 가져오기 (vec)
    vecs = ___

    # matching이 안 된 경우는 0 반환
    if len(vecs) == 0:
        return np.zeros(w2v.vector_size, dtype=np.float32)

    # 전체 단어 임베딩을 평균하여 반환
    return np.mean(vecs, axis=0).astype(np.float32)


if len(items_for_w2v) == 0:
    item_emb_w2v_norm = np.zeros((0, w2v.vector_size), dtype=np.float32)
else:
    item_emb_w2v = np.vstack([embed_tokens_avg(tokenize(doc)) for doc in docs_for_w2v]).astype(np.float32)
    item_emb_w2v_norm = item_emb_w2v / (norm(item_emb_w2v, axis=1, keepdims=True) + 1e-12)

item2row_w2v = {int(item_id): row for row, item_id in enumerate(items_for_w2v)}
candidate_items_w2v = np.asarray(items_for_w2v, dtype=int)
pos_items_by_user = train_df[train_df["rating"] >= 4.0].groupby("u")["i"].apply(list).to_dict()
user_emb_cache_w2v = {}


def build_user_emb_w2v(u: int):
    pos_items = pos_items_by_user.get(int(u), [])
    rows = [item2row_w2v[item] for item in pos_items if item in item2row_w2v]
    if len(rows) == 0:
        return None
    emb = item_emb_w2v_norm[rows].mean(axis=0)
    emb = emb / (norm(emb) + 1e-12)
    return emb.astype(np.float32)


class W2VEmbedRecommender:
    def predict(self, u: int, items):
        u = int(u)
        if u not in user_emb_cache_w2v:
            user_emb_cache_w2v[u] = build_user_emb_w2v(u)
        user_emb = user_emb_cache_w2v[u]
        items = np.asarray(items, dtype=int)
        scores = np.full(len(items), -1e9, dtype=float)
        if user_emb is None:
            return scores

        known = np.array([int(item) in item2row_w2v for item in items], dtype=bool)
        if known.any():
            rows = np.array([item2row_w2v[int(item)] for item in items[known]], dtype=int)
            scores[known] = item_emb_w2v_norm[rows] @ user_emb
        return scores


w2v_model = W2VEmbedRecommender()
w2v_metrics = evaluate_topk_with_candidates(w2v_model, test_df, candidate_items_w2v, k=10, n_neg=100)
w2v_metrics

{'Hit@10': 0.41379310344827586, 'NDCG@10': 0.24174523454912158, 'n_eval': 1160}

## 7. Transformer/BERT 문장 임베딩 기반 추천

문장 임베딩은 리뷰 문서 전체의 의미를 하나의 벡터로 표현합니다. `sentence-transformers`를 사용할 수 있으면 MiniLM 기반 문장 임베딩을 사용하고, 사용할 수 없는 환경에서는 TF-IDF와 SVD 기반 문서 임베딩으로 같은 추천 과정을 진행합니다.

In [ ]:
import hashlib

MAX_ITEMS_SENT = min(1500, len(items_for_text))
items_for_sent = items_for_text[:MAX_ITEMS_SENT]
docs_for_sent = item_docs[:MAX_ITEMS_SENT]


def normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    if matrix.ndim == 1:
        matrix = matrix.reshape(1, -1)
    return matrix / (norm(matrix, axis=1, keepdims=True) + 1e-12)


def hashed_text_embeddings(docs, dim: int = 64):
    embeddings = np.zeros((len(docs), dim), dtype=np.float32)
    for row_idx, doc in enumerate(docs):
        for token in tokenize(doc):
            digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
            value = int.from_bytes(digest, "little", signed=False)
            col_idx = value % dim
            sign = 1.0 if ((value >> 8) & 1) else -1.0
            embeddings[row_idx, col_idx] += sign
    return normalize_rows(embeddings)


def build_sentence_embeddings(docs):
    if len(docs) == 0:
        return np.zeros((0, 1), dtype=np.float32), "empty embedding"

    try:
        from sentence_transformers import SentenceTransformer

        model_kwargs = {}
        try:
            import torch
            model_kwargs["device"] = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            pass

        # 모델 로드
        sentence_model = SentenceTransformer("___", **model_kwargs)

        # 해당 모델 기반 임베딩 계산
        embeddings = sentence_model.encode(
            ___,
            batch_size=32,
            show_progress_bar=False,
            normalize_embeddings=True,
        )

        return np.asarray(embeddings, dtype=np.float32), "SentenceTransformer all-MiniLM-L6-v2"
    except Exception:
        try:
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.decomposition import TruncatedSVD

            vectorizer = TfidfVectorizer(max_features=5000, min_df=1)
            X = vectorizer.fit_transform(docs)
            if X.shape[1] == 0:
                return hashed_text_embeddings(docs), "hashed bag-of-words embedding"
            if X.shape[0] <= 1 or X.shape[1] <= 1:
                return normalize_rows(X.toarray().astype(np.float32)), "TF-IDF document embedding"

            n_components = min(64, X.shape[0] - 1, X.shape[1] - 1)
            n_components = max(1, int(n_components))
            svd = TruncatedSVD(n_components=n_components, random_state=SEED)
            embeddings = svd.fit_transform(X).astype(np.float32)
            return normalize_rows(embeddings), "TF-IDF + SVD document embedding"
        except Exception:
            return hashed_text_embeddings(docs), "hashed bag-of-words embedding"


item_emb_sent, sentence_embedding_source = build_sentence_embeddings(docs_for_sent)
item_emb_sent = normalize_rows(item_emb_sent)

item2row_sent = {int(item_id): row for row, item_id in enumerate(items_for_sent)}
candidate_items_sent = np.asarray(items_for_sent, dtype=int)
pos_items_by_user_sent = train_df[train_df["rating"] >= 4.0].groupby("u")["i"].apply(list).to_dict()
user_emb_cache_sent = {}


def build_user_emb_sent(u: int):
    pos_items = pos_items_by_user_sent.get(int(u), [])
    rows = [item2row_sent[item] for item in pos_items if item in item2row_sent]
    if len(rows) == 0:
        return None
    emb = item_emb_sent[rows].mean(axis=0)
    emb = emb / (norm(emb) + 1e-12)
    return emb.astype(np.float32)


class SentenceEmbedRecommender:
    def predict(self, u: int, items):
        u = int(u)
        if u not in user_emb_cache_sent:
            user_emb_cache_sent[u] = build_user_emb_sent(u)
        user_emb = user_emb_cache_sent[u]
        items = np.asarray(items, dtype=int)
        scores = np.full(len(items), -1e9, dtype=float)
        if user_emb is None:
            return scores

        known = np.array([int(item) in item2row_sent for item in items], dtype=bool)
        if known.any():
            rows = np.array([item2row_sent[int(item)] for item in items[known]], dtype=int)
            scores[known] = item_emb_sent[rows] @ user_emb
        return scores


sentence_model = SentenceEmbedRecommender()
sentence_metrics = evaluate_topk_with_candidates(sentence_model, test_df, candidate_items_sent, k=10, n_neg=100)

print("document embedding source:", sentence_embedding_source)
sentence_metrics

document embedding source: TF-IDF + SVD document embedding


{'Hit@10': 0.4189655172413793, 'NDCG@10': 0.252581775062256, 'n_eval': 1160}